# ARC-v0.1 — Memory-Safe Retrieval Dynamics Observatory

## Why this revision

The original ARC-v0 used `IndexIVFFlat` over the full 5.42M × 384 corpus.
That index stores full float32 document vectors and exhausted standard Colab RAM during add.

ARC-v0.1 changes only the **fixed retrieval environment**:

\[
\boxed{\text{IVF-PQ}(M=32,\;8\text{ bit})}
\]

The ARC research question remains the same:

> Does iterative retrieval exhibit measurable and predictable convergence, wrong attractors,
> saturation, productive exploration, and drift?

## Memory-safe changes

- IVF-PQ instead of IVF-Flat
- 32 B/document PQ payload
- corpus add batch reduced to 10k
- Faiss index persisted to Drive
- existing index is loaded instead of rebuilt
- process RSS monitoring during training/add
- training sample capped at 200k documents
- FIT split only
- TEST remains untouched

## Compute plan

### Stage A
500 FIT queries × small grid × 2 feedback rounds.

### Stage B
Top 4 configurations × 2,000 FIT queries × 4 rounds.

No LLM and no learned controller yet.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
from datetime import datetime
import json, sys, subprocess, time, gc, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260816
DIM = 384

TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

# Fixed memory-safe ANN environment.
NLIST = 4096
NPROBE = 64
PQ_M = 32
PQ_NBITS = 8

TRAIN_DOCS = 200_000
ADD_BATCH = 10_000

# Stage A
GRID_QUERY_COUNT = 500
GRID_KS = [5, 10, 20]
GRID_ALPHAS = [0.1, 0.3, 0.5]
GRID_TEMPS = [0.05, 0.10]

# Stage B
CONFIRM_QUERY_COUNT = 2000
BEST_CONFIGS_TO_CONFIRM = 4

# Dynamics taxonomy thresholds.
JACCARD_STABLE = 0.80
UTILITY_EPS = 1e-6
DRIFT_HIGH = 0.15

ROOT = Path(
    "/content/drive/MyDrive/"
    "hc-rars-fever-5m-untouched-confirmation-v1"
)

CORPUS_MEMMAP = ROOT / "stage1/corpus_embeddings.float16.memmap"
QUERY_EMB = ROOT / "stage1/query_embeddings_v2.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
FIT_QRELS = ROOT / "stage2/fit_qrels_rows.csv"

CACHE_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

INDEX_PATH = CACHE_ROOT / (
    f"fever5m-bge-small-ivfpq-nlist{NLIST}-"
    f"m{PQ_M}-nbits{PQ_NBITS}-seed{SEED}.faiss"
)
INDEX_MANIFEST_PATH = INDEX_PATH.with_suffix(".json")

OUT_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0"
)
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / f"fever5m-observatory-v01-{RUN_ID}"
OUT.mkdir(parents=True, exist_ok=True)

print("Index cache:", INDEX_PATH)
print("Output     :", OUT)


Index cache: /content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss
Output     : /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever5m-observatory-v01-20260815-185737


In [3]:
def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

run([
    sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu==1.12.0",
    "psutil",
    "pyarrow",
    "scikit-learn",
])

import faiss
import psutil

print("Faiss:", faiss.__version__)
print("RAM total GB:", psutil.virtual_memory().total / 1024**3)


$ /usr/bin/python3 -m pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn
Faiss: 1.12.0
RAM total GB: 12.671409606933594


## 1. RAM monitor


In [4]:
PROCESS = psutil.Process(os.getpid())

def ram_status(label=""):
    vm = psutil.virtual_memory()
    rss = PROCESS.memory_info().rss / 1024**3
    print(
        f"[RAM] {label:<28} "
        f"RSS={rss:6.2f} GB | "
        f"available={vm.available/1024**3:6.2f} GB | "
        f"used={vm.percent:5.1f}%"
    )

ram_status("startup")


[RAM] startup                      RSS=  0.17 GB | available= 11.31 GB | used= 10.7%


## 2. Load FEVER corpus/query mappings


In [5]:
size_bytes = CORPUS_MEMMAP.stat().st_size
bytes_per_row = DIM * np.dtype(np.float16).itemsize

assert size_bytes % bytes_per_row == 0
n_docs = size_bytes // bytes_per_row
assert n_docs == 5_416_568

docs = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(n_docs, DIM),
)

query_embeddings = np.load(
    QUERY_EMB,
    mmap_mode="r",
)
assert query_embeddings.shape == (123_142, DIM)

with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [line.strip() for line in f if line.strip()]

assert len(query_ids) == len(query_embeddings)
assert len(set(query_ids)) == len(query_ids)

query_row = {
    qid: i
    for i, qid in enumerate(query_ids)
}

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

fit_ids = [str(x).strip() for x in split["fit_query_ids"]]
dev_ids = [str(x).strip() for x in split["dev_query_ids"]]
test_ids = [str(x).strip() for x in split["test_query_ids"]]

assert split["test_retrieval_performed"] is False
assert split["test_relevance_values_accessed"] is False

missing_fit = [q for q in fit_ids if q not in query_row]
assert not missing_fit, missing_fit[:10]

fit_rows = np.array(
    [query_row[q] for q in fit_ids],
    dtype=np.int64,
)

print("corpus:", docs.shape, docs.dtype)
print("queries:", query_embeddings.shape, query_embeddings.dtype)
print("fit/dev/test:", len(fit_ids), len(dev_ids), len(test_ids))

ram_status("after mappings")


corpus: (5416568, 384) float16
queries: (123142, 384) float32
fit/dev/test: 20000 6666 6666
[RAM] after mappings               RSS=  0.19 GB | available= 11.57 GB | used=  8.7%


In [6]:
fit_qrels_df = pd.read_csv(FIT_QRELS)
fit_qrels_df["query-id"] = fit_qrels_df["query-id"].astype(str)

def make_qrels_map(df):
    out = {}
    for qid, g in df.groupby("query-id"):
        rel = g[g["score"] > 0]["corpus-row"].astype(np.int64)
        out[str(qid)] = set(rel.tolist())
    return out

fit_qrels = make_qrels_map(fit_qrels_df)

print("FIT qrels queries:", len(fit_qrels))
display(fit_qrels_df.head())


FIT qrels queries: 20000


,query-id,corpus-id,score,corpus-row
0,150448,Roman_Atwood,1,4222138
1,129629,Prisoners_of_War_(TV_series),1,3911864
2,129629,Homeland_(TV_series),1,2260917
3,33078,Boston_Celtics,1,841120
4,188923,Chad,1,1059381


## 3. Build or load persistent IVF-PQ index

The full index is serialized to Drive.

If the runtime disconnects after a successful build, rerunning the notebook loads the
existing `.faiss` file instead of rebuilding 5.4M documents.

Expected code payload:

\[
M\times nbits/8 = 32 \text{ B/document}
\]

or approximately 173 MB of PQ codes for 5.42M documents, plus inverted-list IDs and
index metadata.


In [7]:
def index_manifest():
    return {
        "schema_version": 1,
        "dataset": "FEVER",
        "corpus_rows": int(n_docs),
        "dimension": DIM,
        "metric": "inner_product",
        "nlist": NLIST,
        "nprobe_runtime": NPROBE,
        "M": PQ_M,
        "nbits": PQ_NBITS,
        "seed": SEED,
        "train_docs": TRAIN_DOCS,
        "encoder": "BAAI/bge-small-en-v1.5",
    }

def manifest_matches(path):
    if not path.is_file():
        return False
    try:
        old = json.loads(path.read_text(encoding="utf-8"))
        expected = index_manifest()
        keys = [
            "dataset","corpus_rows","dimension","metric",
            "nlist","M","nbits","seed","train_docs","encoder"
        ]
        return all(old.get(k) == expected.get(k) for k in keys)
    except Exception:
        return False

if INDEX_PATH.is_file() and manifest_matches(INDEX_MANIFEST_PATH):
    print("Loading cached IVF-PQ index...")
    ram_status("before index load")
    index = faiss.read_index(str(INDEX_PATH))
    index.nprobe = NPROBE
    print("Loaded:", INDEX_PATH)
    print("ntotal:", index.ntotal)
    ram_status("after index load")

else:
    print("Building new IVF-PQ index...")
    rng = np.random.default_rng(SEED)

    train_doc_ids = rng.choice(
        n_docs,
        size=min(TRAIN_DOCS, n_docs),
        replace=False,
    )

    ram_status("before training sample")

    train_x = np.ascontiguousarray(
        np.asarray(
            docs[train_doc_ids],
            dtype=np.float32,
        )
    )

    ram_status("training sample ready")

    coarse = faiss.IndexFlatIP(DIM)

    index = faiss.IndexIVFPQ(
        coarse,
        DIM,
        NLIST,
        PQ_M,
        PQ_NBITS,
        faiss.METRIC_INNER_PRODUCT,
    )

    print("Training IVF-PQ...")
    t0 = time.time()
    index.train(train_x)
    print("training minutes:", (time.time()-t0)/60)

    del train_x
    gc.collect()
    ram_status("after IVF-PQ training")

    print("Streaming corpus into IVF-PQ...")
    t0 = time.time()

    for start in range(0, n_docs, ADD_BATCH):
        end = min(start + ADD_BATCH, n_docs)

        xb = np.asarray(
            docs[start:end],
            dtype=np.float32,
        )

        index.add(
            np.ascontiguousarray(xb)
        )

        del xb

        if end % 250_000 < ADD_BATCH or end == n_docs:
            gc.collect()
            print(
                f"added {end:,}/{n_docs:,} "
                f"({100*end/n_docs:.1f}%)"
            )
            ram_status(f"add {end:,}")

    print("add minutes:", (time.time()-t0)/60)

    assert index.ntotal == n_docs

    index.nprobe = NPROBE

    print("Saving index to Drive...")
    faiss.write_index(index, str(INDEX_PATH))

    INDEX_MANIFEST_PATH.write_text(
        json.dumps(index_manifest(), indent=2),
        encoding="utf-8",
    )

    print("Saved:", INDEX_PATH)
    ram_status("after index save")

assert index.ntotal == n_docs
index.nprobe = NPROBE
print("nlist:", index.nlist, "nprobe:", index.nprobe)


Building new IVF-PQ index...
[RAM] before training sample       RSS=  0.21 GB | available= 11.17 GB | used= 11.9%
[RAM] training sample ready        RSS=  4.01 GB | available= 10.58 GB | used= 16.5%
Training IVF-PQ...
training minutes: 2.5705573201179504
[RAM] after IVF-PQ training        RSS=  3.76 GB | available= 10.90 GB | used= 14.0%
Streaming corpus into IVF-PQ...
added 250,000/5,416,568 (4.6%)
[RAM] add 250,000                  RSS=  3.81 GB | available= 10.86 GB | used= 14.3%
added 500,000/5,416,568 (9.2%)
[RAM] add 500,000                  RSS=  3.85 GB | available= 10.86 GB | used= 14.3%
added 750,000/5,416,568 (13.8%)
[RAM] add 750,000                  RSS=  3.86 GB | available= 10.87 GB | used= 14.2%
added 1,000,000/5,416,568 (18.5%)
[RAM] add 1,000,000                RSS=  3.88 GB | available= 10.89 GB | used= 14.0%
added 1,250,000/5,416,568 (23.1%)
[RAM] add 1,250,000                RSS=  3.91 GB | available= 10.90 GB | used= 14.0%
added 1,500,000/5,416,568 (27.7%)
[RAM] a

## 4. Quick retrieval sanity check


In [8]:
def normalize_rows(x):
    x = np.asarray(x, np.float32)
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norm, 1e-12)

sanity_q = normalize_rows(
    np.asarray(query_embeddings[fit_rows[:32]], dtype=np.float32)
)

scores_sanity, ids_sanity = index.search(
    np.ascontiguousarray(sanity_q, np.float32),
    TOP_RETRIEVE,
)

assert scores_sanity.shape == (32, TOP_RETRIEVE)
assert ids_sanity.shape == (32, TOP_RETRIEVE)
assert np.all(ids_sanity >= 0)

print("sanity search passed")
print("top IDs:", ids_sanity[0,:10])
print("top scores:", scores_sanity[0,:10])
ram_status("after sanity search")


sanity search passed
top IDs: [5240624 4893296 4957180 5375481 4677388 4660024 4960853 4987816 4953448
 5012663]
top scores: [0.70614356 0.69157654 0.68730813 0.67904437 0.67546743 0.67308605
 0.6674409  0.66637623 0.6634728  0.6628838 ]
[RAM] after sanity search          RSS=  4.35 GB | available= 10.72 GB | used= 15.4%


## 5. Retrieval-state metrics


In [9]:
def evaluate_one(qid, ranked_ids, k=10):
    relset = fit_qrels.get(str(qid), set())
    ranked = ranked_ids[:k]

    hits = np.array(
        [1.0 if int(d) in relset else 0.0 for d in ranked],
        dtype=np.float32,
    )

    recall = float(
        hits.sum() / max(len(relset), 1)
    )

    pos = np.flatnonzero(hits)
    mrr = float(1.0/(pos[0]+1)) if len(pos) else 0.0

    discounts = 1.0 / np.log2(np.arange(2, k+2))
    dcg = float((hits * discounts).sum())
    ideal_n = min(len(relset), k)
    idcg = float(discounts[:ideal_n].sum()) if ideal_n else 0.0
    ndcg = float(dcg/idcg) if idcg > 0 else 0.0

    return recall, mrr, ndcg

def jaccard(a, b):
    a = set(map(int, a))
    b = set(map(int, b))
    return len(a & b) / max(len(a | b), 1)

def rank_overlap(a, b, k=10):
    return len(
        set(map(int, a[:k]))
        & set(map(int, b[:k]))
    ) / k

def score_entropy(scores, temperature=0.05):
    s = np.asarray(scores, np.float64)
    z = s / temperature
    z -= z.max()
    p = np.exp(np.clip(z, -60, 60))
    p /= np.maximum(p.sum(), 1e-12)
    return float(
        -np.sum(p * np.log(np.maximum(p, 1e-12)))
    )

def candidate_dispersion(candidate_vecs):
    x = normalize_rows(candidate_vecs)
    centroid = normalize_rows(
        x.mean(axis=0, keepdims=True)
    )[0]
    return float(
        np.mean(1.0 - x @ centroid)
    )

def retrieval_state_features(
    q0,
    qt,
    scores,
    ids,
    prev_ids=None,
):
    cand_vecs = np.asarray(
        docs[np.asarray(ids[:20], np.int64)],
        dtype=np.float32,
    )

    anchor_sim = float(
        np.dot(q0, qt)
        / max(
            np.linalg.norm(q0) * np.linalg.norm(qt),
            1e-12,
        )
    )

    out = {
        "query_drift": 1.0-anchor_sim,
        "score_entropy": score_entropy(scores[:20]),
        "score_margin_1_2": float(scores[0]-scores[1]),
        "score_margin_10_11": float(scores[9]-scores[10]),
        "candidate_dispersion": candidate_dispersion(cand_vecs),
        "candidate_jaccard": np.nan,
        "rank_overlap@10": np.nan,
    }

    if prev_ids is not None:
        out["candidate_jaccard"] = jaccard(
            ids[:TOP_RETRIEVE],
            prev_ids[:TOP_RETRIEVE],
        )
        out["rank_overlap@10"] = rank_overlap(
            ids,
            prev_ids,
            TOP_K,
        )

    return out


## 6. Feedback operators


In [10]:
def feedback_vector(
    ids,
    scores,
    k,
    method,
    temperature=None,
):
    ids = np.asarray(ids[:k], np.int64)

    x = np.asarray(
        docs[ids],
        dtype=np.float32,
    )

    if method == "mean":
        f = x.mean(axis=0)

    elif method == "softmax":
        s = np.asarray(scores[:k], np.float64)
        z = s / float(temperature)
        z -= z.max()
        w = np.exp(np.clip(z, -60, 60))
        w /= np.maximum(w.sum(), 1e-12)
        f = (x * w[:, None]).sum(axis=0)

    else:
        raise ValueError(method)

    f = f.astype(np.float32)
    return f / max(np.linalg.norm(f), 1e-12)

def anchored_update(
    q0,
    feedback,
    alpha,
):
    q = (
        (1.0-alpha)*q0
        + alpha*feedback
    ).astype(np.float32)

    return q / max(
        np.linalg.norm(q),
        1e-12,
    )


## 7. Batched trajectory runner


In [11]:
def run_trajectory(
    query_indices,
    config,
    max_rounds=MAX_ROUNDS,
):
    query_indices = np.asarray(
        query_indices,
        np.int64,
    )

    q0 = normalize_rows(
        np.asarray(
            query_embeddings[query_indices],
            dtype=np.float32,
        )
    )

    qt = q0.copy()
    qids = [query_ids[i] for i in query_indices]

    rows = []
    prev_ids = [None] * len(query_indices)

    for t in range(max_rounds+1):
        print(
            f"  iteration {t}/{max_rounds} | "
            f"queries={len(query_indices)}"
        )

        scores, ids = index.search(
            np.ascontiguousarray(qt, np.float32),
            TOP_RETRIEVE,
        )

        next_q = np.empty_like(qt)

        for i in range(len(query_indices)):
            recall, mrr, ndcg = evaluate_one(
                qids[i],
                ids[i],
                TOP_K,
            )

            state = retrieval_state_features(
                q0[i],
                qt[i],
                scores[i],
                ids[i],
                prev_ids[i],
            )

            rows.append({
                "query_row": int(query_indices[i]),
                "query_id": qids[i],
                "iteration": t,
                "method": config["method"],
                "k_feedback": int(config["k"]),
                "alpha": float(config["alpha"]),
                "temperature": (
                    float(config["temperature"])
                    if config.get("temperature") is not None
                    else np.nan
                ),
                "recall@10": recall,
                "mrr@10": mrr,
                "ndcg@10": ndcg,
                **state,
            })

            if t < max_rounds:
                fb = feedback_vector(
                    ids[i],
                    scores[i],
                    config["k"],
                    config["method"],
                    config.get("temperature"),
                )

                next_q[i] = anchored_update(
                    q0[i],
                    fb,
                    config["alpha"],
                )

            prev_ids[i] = ids[i].copy()

        if t < max_rounds:
            qt = next_q

        ram_status(f"trajectory iter {t}")

    return pd.DataFrame(rows)


## 8. Stage A — 500-query screen


In [12]:
fit_rng = np.random.default_rng(SEED+1)

grid_query_rows = fit_rng.choice(
    fit_rows,
    size=GRID_QUERY_COUNT,
    replace=False,
)

configs = []

for k in GRID_KS:
    for alpha in GRID_ALPHAS:

        configs.append({
            "method": "mean",
            "k": k,
            "alpha": alpha,
            "temperature": None,
        })

        for temp in GRID_TEMPS:
            configs.append({
                "method": "softmax",
                "k": k,
                "alpha": alpha,
                "temperature": temp,
            })

print("Grid configs:", len(configs))

grid_frames = []

for ci, cfg in enumerate(configs):
    print("="*90)
    print(
        f"CONFIG {ci+1}/{len(configs)}",
        cfg,
    )

    df = run_trajectory(
        grid_query_rows,
        cfg,
        max_rounds=2,
    )

    grid_frames.append(df)

grid_df = pd.concat(
    grid_frames,
    ignore_index=True,
)

print("Stage A rows:", len(grid_df))


Grid configs: 27
CONFIG 1/27 {'method': 'mean', 'k': 5, 'alpha': 0.1, 'temperature': None}
  iteration 0/2 | queries=500
[RAM] trajectory iter 0            RSS=  4.38 GB | available= 10.75 GB | used= 15.2%
  iteration 1/2 | queries=500
[RAM] trajectory iter 1            RSS=  4.38 GB | available= 10.75 GB | used= 15.2%
  iteration 2/2 | queries=500
[RAM] trajectory iter 2            RSS=  4.38 GB | available= 10.74 GB | used= 15.2%
CONFIG 2/27 {'method': 'softmax', 'k': 5, 'alpha': 0.1, 'temperature': 0.05}
  iteration 0/2 | queries=500
[RAM] trajectory iter 0            RSS=  4.38 GB | available= 10.74 GB | used= 15.3%
  iteration 1/2 | queries=500
[RAM] trajectory iter 1            RSS=  4.38 GB | available= 10.74 GB | used= 15.3%
  iteration 2/2 | queries=500
[RAM] trajectory iter 2            RSS=  4.38 GB | available= 10.75 GB | used= 15.2%
CONFIG 3/27 {'method': 'softmax', 'k': 5, 'alpha': 0.1, 'temperature': 0.1}
  iteration 0/2 | queries=500
[RAM] trajectory iter 0            R

In [13]:
def summarize_config_trajectory(df):
    rows = []

    group_cols = [
        "method",
        "k_feedback",
        "alpha",
        "temperature",
    ]

    for key, g in df.groupby(
        group_cols,
        dropna=False,
    ):
        pivot = g.pivot(
            index="query_id",
            columns="iteration",
            values="ndcg@10",
        )

        base = pivot[0]
        final_iter = max(pivot.columns)
        final = pivot[final_iter]
        oracle = pivot.max(axis=1)

        rows.append({
            "method": key[0],
            "k_feedback": key[1],
            "alpha": key[2],
            "temperature": key[3],
            "base_ndcg@10": float(base.mean()),
            "final_ndcg@10": float(final.mean()),
            "oracle_ndcg@10": float(oracle.mean()),
            "final_minus_base": float((final-base).mean()),
            "oracle_minus_base": float((oracle-base).mean()),
            "oracle_minus_final": float((oracle-final).mean()),
            "harm_rate_final": float(np.mean(final < base)),
            "improve_rate_final": float(np.mean(final > base)),
        })

    return pd.DataFrame(rows)

grid_summary = (
    summarize_config_trajectory(grid_df)
    .sort_values(
        ["oracle_minus_base","final_ndcg@10"],
        ascending=[False,False],
    )
    .reset_index(drop=True)
)

display(grid_summary.head(20))


,method,k_feedback,alpha,temperature,base_ndcg@10,final_ndcg@10,oracle_ndcg@10,final_minus_base,oracle_minus_base,oracle_minus_final,harm_rate_final,improve_rate_final
0,softmax,5,0.5,0.10,0.14006,0.112756,0.142413,-0.027303,0.002353,0.029657,0.062,0.008
1,softmax,5,0.5,0.05,0.14006,0.121355,0.142387,-0.018705,0.002327,0.021033,0.046,0.008
2,softmax,20,0.3,0.10,0.14006,0.124124,0.142274,-0.015936,0.002214,0.018150,0.038,0.006
3,mean,20,0.3,NaN,0.14006,0.121798,0.142274,-0.018262,0.002214,0.020477,0.038,0.004
4,mean,10,0.5,NaN,0.14006,0.103474,0.142274,-0.036586,0.002214,0.038800,0.070,0.002
5,softmax,20,0.5,0.05,0.14006,0.102826,0.142274,-0.037234,0.002214,0.039448,0.072,0.004
6,softmax,20,0.5,0.10,0.14006,0.099191,0.142274,-0.040869,0.002214,0.043083,0.074,0.002
7,softmax,10,0.5,0.05,0.14006,0.109755,0.142167,-0.030304,0.002107,0.032412,0.062,0.006
8,softmax,10,0.3,0.10,0.14006,0.126565,0.141675,-0.013495,0.001615,0.015110,0.034,0.004
9,softmax,20,0.3,0.05,0.14006,0.124957,0.141675,-0.015103,0.001615,0.016718,0.036,0.004


## 9. Stage B — 2,000-query mechanism confirmation


In [14]:
best_cfg_rows = grid_summary.head(
    BEST_CONFIGS_TO_CONFIRM
)

best_configs = []

for _, r in best_cfg_rows.iterrows():
    best_configs.append({
        "method": str(r["method"]),
        "k": int(r["k_feedback"]),
        "alpha": float(r["alpha"]),
        "temperature": (
            None
            if pd.isna(r["temperature"])
            else float(r["temperature"])
        ),
    })

print("Selected configs:")
for c in best_configs:
    print(c)

confirm_query_rows = fit_rng.choice(
    fit_rows,
    size=CONFIRM_QUERY_COUNT,
    replace=False,
)

confirm_frames = []

for ci, cfg in enumerate(best_configs):
    print("="*90)
    print(
        f"CONFIRM {ci+1}/{len(best_configs)}",
        cfg,
    )

    confirm_frames.append(
        run_trajectory(
            confirm_query_rows,
            cfg,
            max_rounds=MAX_ROUNDS,
        )
    )

confirm_df = pd.concat(
    confirm_frames,
    ignore_index=True,
)

print("Stage B rows:", len(confirm_df))


Selected configs:
{'method': 'softmax', 'k': 5, 'alpha': 0.5, 'temperature': 0.1}
{'method': 'softmax', 'k': 5, 'alpha': 0.5, 'temperature': 0.05}
{'method': 'softmax', 'k': 20, 'alpha': 0.3, 'temperature': 0.1}
{'method': 'mean', 'k': 20, 'alpha': 0.3, 'temperature': None}
CONFIRM 1/4 {'method': 'softmax', 'k': 5, 'alpha': 0.5, 'temperature': 0.1}
  iteration 0/4 | queries=2000
[RAM] trajectory iter 0            RSS=  4.44 GB | available= 10.59 GB | used= 16.4%
  iteration 1/4 | queries=2000
[RAM] trajectory iter 1            RSS=  4.45 GB | available= 10.61 GB | used= 16.3%
  iteration 2/4 | queries=2000
[RAM] trajectory iter 2            RSS=  4.45 GB | available= 10.61 GB | used= 16.3%
  iteration 3/4 | queries=2000
[RAM] trajectory iter 3            RSS=  4.45 GB | available= 10.60 GB | used= 16.3%
  iteration 4/4 | queries=2000
[RAM] trajectory iter 4            RSS=  4.45 GB | available= 10.60 GB | used= 16.3%
CONFIRM 2/4 {'method': 'softmax', 'k': 5, 'alpha': 0.5, 'temperature'

## 10. Retrieval trajectory taxonomy


In [15]:
group_cols = [
    "query_id",
    "method",
    "k_feedback",
    "alpha",
    "temperature",
]

trajectory_labels = []

for key, g in confirm_df.groupby(
    group_cols,
    dropna=False,
):
    g = g.sort_values(
        "iteration"
    ).reset_index(drop=True)

    base_u = float(g.iloc[0]["ndcg@10"])
    final_u = float(g.iloc[-1]["ndcg@10"])
    final_j = float(g.iloc[-1]["candidate_jaccard"])
    final_drift = float(g.iloc[-1]["query_drift"])

    delta = final_u-base_u

    if (
        not np.isnan(final_j)
        and final_j >= JACCARD_STABLE
    ):
        if delta > UTILITY_EPS:
            label = "correct_convergence"
        elif delta < -UTILITY_EPS:
            label = "wrong_attractor"
        else:
            label = "saturation"

    elif (
        delta < -UTILITY_EPS
        and final_drift >= DRIFT_HIGH
    ):
        label = "drift"

    elif delta > UTILITY_EPS:
        label = "productive_exploration"

    else:
        label = "unstable_no_gain"

    js = (
        g["candidate_jaccard"]
        .dropna()
        .to_numpy()
    )

    oscillation = bool(
        len(js) >= 3
        and np.max(js[:-1]) >= JACCARD_STABLE
        and js[-1] < 0.5
    )

    trajectory_labels.append({
        "query_id": key[0],
        "method": key[1],
        "k_feedback": key[2],
        "alpha": key[3],
        "temperature": key[4],
        "base_ndcg@10": base_u,
        "final_ndcg@10": final_u,
        "delta_ndcg@10": delta,
        "final_candidate_jaccard": final_j,
        "final_query_drift": final_drift,
        "trajectory_label": label,
        "oscillation_flag": oscillation,
    })

labels_df = pd.DataFrame(
    trajectory_labels
)

display(
    labels_df["trajectory_label"]
    .value_counts(normalize=True)
    .rename("fraction")
    .to_frame()
)

print(
    "Oscillation fraction:",
    labels_df["oscillation_flag"].mean(),
)


,fraction
trajectory_label,
saturation,0.889375
wrong_attractor,0.054625
unstable_no_gain,0.047375
correct_convergence,0.008000
productive_exploration,0.000625


Oscillation fraction: 0.00025


## 11. Oracle stopping headroom


In [16]:
oracle_rows = []

for key, g in confirm_df.groupby(
    group_cols,
    dropna=False,
):
    g = g.sort_values("iteration")

    utilities = g["ndcg@10"].to_numpy()
    iterations = g["iteration"].to_numpy()

    best_pos = int(
        np.argmax(utilities)
    )

    oracle_rows.append({
        "query_id": key[0],
        "method": key[1],
        "k_feedback": key[2],
        "alpha": key[3],
        "temperature": key[4],
        "base_ndcg@10": float(utilities[0]),
        "final_ndcg@10": float(utilities[-1]),
        "oracle_ndcg@10": float(utilities[best_pos]),
        "oracle_iteration": int(iterations[best_pos]),
        "oracle_minus_base": float(
            utilities[best_pos]-utilities[0]
        ),
        "oracle_minus_final": float(
            utilities[best_pos]-utilities[-1]
        ),
    })

oracle_df = pd.DataFrame(
    oracle_rows
)

oracle_summary = (
    oracle_df
    .groupby(
        [
            "method",
            "k_feedback",
            "alpha",
            "temperature",
        ],
        dropna=False,
    )
    .agg(
        base_ndcg=("base_ndcg@10","mean"),
        final_ndcg=("final_ndcg@10","mean"),
        oracle_ndcg=("oracle_ndcg@10","mean"),
        oracle_minus_base=("oracle_minus_base","mean"),
        oracle_minus_final=("oracle_minus_final","mean"),
        mean_oracle_iteration=("oracle_iteration","mean"),
    )
    .reset_index()
    .sort_values(
        "oracle_minus_base",
        ascending=False,
    )
)

display(oracle_summary)


,method,k_feedback,alpha,temperature,base_ndcg,final_ndcg,oracle_ndcg,oracle_minus_base,oracle_minus_final,mean_oracle_iteration
1,softmax,5,0.5,0.05,0.148556,0.122297,0.151646,0.003091,0.029349,0.0185
2,softmax,5,0.5,0.10,0.148556,0.121579,0.151288,0.002732,0.029708,0.0145
3,softmax,20,0.3,0.10,0.148556,0.132391,0.150644,0.002088,0.018253,0.0140
0,mean,20,0.3,NaN,0.148556,0.131698,0.150598,0.002042,0.018899,0.0110


## 12. Predict next-step improvement from retrieval state


In [17]:
step_rows = []

for key, g in confirm_df.groupby(
    group_cols,
    dropna=False,
):
    g = g.sort_values(
        "iteration"
    ).reset_index(drop=True)

    for i in range(len(g)-1):
        cur = g.iloc[i]
        nxt = g.iloc[i+1]

        step_rows.append({
            "query_id": cur["query_id"],
            "method": cur["method"],
            "iteration": int(cur["iteration"]),
            "query_drift": float(cur["query_drift"]),
            "candidate_jaccard": (
                0.0
                if pd.isna(cur["candidate_jaccard"])
                else float(cur["candidate_jaccard"])
            ),
            "rank_overlap@10": (
                0.0
                if pd.isna(cur["rank_overlap@10"])
                else float(cur["rank_overlap@10"])
            ),
            "score_entropy": float(cur["score_entropy"]),
            "score_margin_1_2": float(cur["score_margin_1_2"]),
            "score_margin_10_11": float(cur["score_margin_10_11"]),
            "candidate_dispersion": float(cur["candidate_dispersion"]),
            "delta_next_ndcg": float(
                nxt["ndcg@10"]-cur["ndcg@10"]
            ),
            "next_improves": int(
                nxt["ndcg@10"]
                > cur["ndcg@10"] + UTILITY_EPS
            ),
            "next_harms": int(
                nxt["ndcg@10"]
                < cur["ndcg@10"] - UTILITY_EPS
            ),
        })

step_df = pd.DataFrame(
    step_rows
)

print("step rows   :", len(step_df))
print("improve rate:", step_df["next_improves"].mean())
print("harm rate   :", step_df["next_harms"].mean())


step rows   : 32000
improve rate: 0.00375
harm rate   : 0.02390625


In [18]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols = [
    "iteration",
    "query_drift",
    "candidate_jaccard",
    "rank_overlap@10",
    "score_entropy",
    "score_margin_1_2",
    "score_margin_10_11",
    "candidate_dispersion",
]

X = step_df[feature_cols].to_numpy(
    dtype=np.float32
)
y = step_df["next_improves"].to_numpy(
    dtype=np.int32
)
groups = step_df["query_id"].astype(str).to_numpy()

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
    ),
)

pred = cross_val_predict(
    model,
    X,
    y,
    groups=groups,
    cv=GroupKFold(n_splits=5),
    method="predict_proba",
)[:,1]

auc = roc_auc_score(
    y,
    pred,
)
ap = average_precision_score(
    y,
    pred,
)

print("ROC-AUC  :", auc)
print("AP       :", ap)
print("base rate:", y.mean())


ROC-AUC  : 0.7701276923881221
AP       : 0.017677321172610005
base rate: 0.00375


## 13. ARC-v0.1 decision


In [19]:
best_oracle = oracle_summary.iloc[0]

oracle_headroom = float(
    best_oracle["oracle_minus_base"]
)
stopping_headroom = float(
    best_oracle["oracle_minus_final"]
)

wrong_attractor_rate = float(
    np.mean(
        labels_df["trajectory_label"]
        == "wrong_attractor"
    )
)

correct_convergence_rate = float(
    np.mean(
        labels_df["trajectory_label"]
        == "correct_convergence"
    )
)

print("=== ARC-v0.1 RETRIEVAL DYNAMICS OBSERVATORY ===")
print(f"Best oracle gain over base nDCG@10 : {oracle_headroom:+.6f}")
print(f"Oracle stopping gain over final     : {stopping_headroom:+.6f}")
print(f"Correct-convergence rate            : {correct_convergence_rate:.3%}")
print(f"Wrong-attractor rate                : {wrong_attractor_rate:.3%}")
print(f"Next-step improvement ROC-AUC       : {auc:.4f}")
print(f"Next-step improvement AP            : {ap:.4f}")
print()

phenomenon_exists = (
    wrong_attractor_rate >= 0.02
    or stopping_headroom >= 0.005
)

predictable = auc >= 0.60
headroom_exists = oracle_headroom >= 0.01

if (
    phenomenon_exists
    and predictable
    and headroom_exists
):
    decision = (
        "STRONG ARC GO: iterative retrieval has meaningful oracle headroom, "
        "nontrivial failure modes, and retrieval-state features predict next-step "
        "improvement. Proceed to ARC-v1 state controller."
    )

elif phenomenon_exists and headroom_exists:
    decision = (
        "MECHANISM GO, POLICY NOT READY: iterative retrieval has real headroom/"
        "failure modes, but current state features are not predictive enough."
    )

elif headroom_exists:
    decision = (
        "HEADROOM ONLY: iterative feedback can help, but convergence failure modes "
        "are not yet strongly measurable."
    )

else:
    decision = (
        "ARC NO-GO: iterative retrieval offers too little oracle utility headroom "
        "under this setup."
    )

print("DECISION:", decision)


=== ARC-v0.1 RETRIEVAL DYNAMICS OBSERVATORY ===
Best oracle gain over base nDCG@10 : +0.003091
Oracle stopping gain over final     : +0.029349
Correct-convergence rate            : 0.800%
Wrong-attractor rate                : 5.463%
Next-step improvement ROC-AUC       : 0.7701
Next-step improvement AP            : 0.0177

DECISION: ARC NO-GO: iterative retrieval offers too little oracle utility headroom under this setup.


## 14. Save evidence


In [20]:
grid_df.to_parquet(
    OUT/"stageA_grid_trajectories.parquet",
    index=False,
)

grid_summary.to_csv(
    OUT/"stageA_grid_summary.csv",
    index=False,
)

confirm_df.to_parquet(
    OUT/"stageB_confirm_trajectories.parquet",
    index=False,
)

labels_df.to_csv(
    OUT/"trajectory_labels.csv",
    index=False,
)

oracle_df.to_csv(
    OUT/"oracle_stopping_per_query.csv",
    index=False,
)

oracle_summary.to_csv(
    OUT/"oracle_stopping_summary.csv",
    index=False,
)

step_df.to_parquet(
    OUT/"step_prediction_dataset.parquet",
    index=False,
)

report = {
    "design": {
        "dataset": "FEVER",
        "corpus_rows": int(n_docs),
        "encoder": "BAAI/bge-small-en-v1.5",
        "split": "FIT only",
        "retriever": "IVF-PQ",
        "nlist": NLIST,
        "nprobe": NPROBE,
        "M": PQ_M,
        "nbits": PQ_NBITS,
        "grid_queries": GRID_QUERY_COUNT,
        "confirm_queries": CONFIRM_QUERY_COUNT,
        "max_rounds": MAX_ROUNDS,
        "test_retrieval_performed": False,
    },
    "index_path": str(INDEX_PATH),
    "best_oracle_config": best_oracle.to_dict(),
    "oracle_headroom_over_base": oracle_headroom,
    "oracle_stopping_headroom_over_final": stopping_headroom,
    "correct_convergence_rate": correct_convergence_rate,
    "wrong_attractor_rate": wrong_attractor_rate,
    "next_improvement_roc_auc": float(auc),
    "next_improvement_average_precision": float(ap),
    "decision": decision,
}

(OUT/"report.json").write_text(
    json.dumps(
        report,
        indent=2,
        default=float,
    ),
    encoding="utf-8",
)

print("Saved:", OUT)
print("Report:", OUT/"report.json")
print("Index cache retained at:", INDEX_PATH)


Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever5m-observatory-v01-20260815-185737
Report: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever5m-observatory-v01-20260815-185737/report.json
Index cache retained at: /content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss
